In [1]:
from pprint import pprint
import datasets as datasets_lib
import grain
import pandas as pd
import os
import fsspec
import numpy as np

import transformers
from tunix.generate import mappings

Dataset = datasets_lib.Dataset
AutoTokenizer = transformers.AutoTokenizer

from absl import logging as absl_logging

import logging
import sys

# ====== Logging Configuration ======
# 1. Force absl to use python logging
absl_logging.use_python_logging()

# 2. Configure the root logger
logging.basicConfig(
    stream=sys.stdout,
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - [%(name)s] %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
    force=True,
)

# 3. Explicitly set levels for relevant loggers
logging.getLogger().setLevel(logging.INFO)
logging.getLogger("absl").setLevel(logging.INFO)

# 4. Set absl verbosity
absl_logging.set_verbosity(absl_logging.INFO)
absl_logging.set_stderrthreshold("info")

print("Logging configured at INFO level.")
import os

os.environ["JAX_COMPILER_CACHE_TGTS"] = ""

/mnt/disks/linchai-data/miniconda3/envs/tunix/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Logging configured at INFO level.


In [2]:
try:
  from GOOGLE_INTERNAL_PACKAGE_PATH.pyglib import gfile
  from etils import ecolab

  cm = ecolab.adhoc(
      source=ecolab.FROM_NOTEBOOK_OR_HEAD,
      reload="tunix",
      behavior="preferred",
      cell_autoreload=True,
  )

  file_open = gfile.Open

  NOTEBOOK_ENV = "g3"
except Exception:
  NOTEBOOK_ENV = "git"

  import contextlib
  cm = contextlib.nullcontext()

  file_open = fsspec.open

In [3]:
with cm:
  from tunix.models.gemma4 import model as gemma4_lib
  from tunix.models.gemma4 import params_safetensors as gemma4_params_lib
  from tunix.generate import sampler as sampler_lib
  from tunix.utils import math_utils
# %%
from typing import Any, Dict, Optional
import jax
from jax import numpy as jnp
from flax import nnx
import orbax.checkpoint as ocp
from tqdm.auto import tqdm
import re

In [4]:
MODEL_PATH_PREFIX = "gs://tunix/models"
MODEL_MAPPING = {
    "google/gemma-4-E2B-it": (
      gemma4_lib.ModelConfig.gemma4_e2b(),
      "/mnt/disks/linchai-data/huggingface/hub/models--google--gemma-4-E2B-it/snapshots/905e84b50c4d2a365ebde34e685027578e6728db",
    ),
    
}

In [5]:
import os
os.environ["TPU_LOG_DIR"] = "~/my_tpu_logs"
os.environ["SKIP_JAX_PRECOMPILE"] = "True"

In [6]:

model_version = "google/gemma-4-E2B-it"
model_config, model_path = MODEL_MAPPING[model_version]

tokenizer = AutoTokenizer.from_pretrained(model_version)

2026-05-28 03:02:19 - INFO - [httpx] HTTP Request: HEAD https://huggingface.co/google/gemma-4-E2B-it/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-05-28 03:02:19 - INFO - [httpx] HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/google/gemma-4-E2B-it/905e84b50c4d2a365ebde34e685027578e6728db/config.json "HTTP/1.1 200 OK"


2026-05-28 03:02:19 - INFO - [httpx] HTTP Request: HEAD https://huggingface.co/google/gemma-4-E2B-it/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
2026-05-28 03:02:19 - INFO - [httpx] HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/google/gemma-4-E2B-it/905e84b50c4d2a365ebde34e685027578e6728db/tokenizer_config.json "HTTP/1.1 200 OK"
2026-05-28 03:02:19 - INFO - [httpx] HTTP Request: GET https://huggingface.co/api/models/google/gemma-4-E2B-it/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
2026-05-28 03:02:19 - INFO - [httpx] HTTP Request: GET https://huggingface.co/api/models/google/gemma-4-E2B-it/tree/main?recursive=true&expand=false "HTTP/1.1 200 OK"
2026-05-28 03:02:22 - INFO - [httpx] HTTP Request: GET https://huggingface.co/api/models/google/gemma-4-E2B-it "HTTP/1.1 200 OK"


In [7]:
trainer_mesh = jax.sharding.Mesh(np.array(jax.devices()).reshape(1, 4), axis_names=["fsdp", "tp"])
rollout_mesh = jax.sharding.Mesh(np.array(jax.devices()).reshape(1, 4), axis_names=["fsdp", "tp"])

Could not access ~/my_tpu_logs: not found


In [8]:
print("Loading model from safe tensors...")
with trainer_mesh:
  trainer_model = gemma4_params_lib.create_model_from_safe_tensors(file_dir=model_path, config=model_config, mesh=trainer_mesh)
  ref_model = gemma4_params_lib.create_model_from_safe_tensors(file_dir=model_path, config=model_config, mesh=trainer_mesh)

import jax
jax.clear_caches()

Loading model from safe tensors...


2026-05-28 03:02:37 - WARNING - [absl] Skipped loading 1411 keys because they could not be mapped to model weights. This may be expected, for example when loading only text weights from a multimodal checkpoint. Missing keys: [model.audio_tower.layers.0.feed_forward1.ffw_layer_1.input_max, model.audio_tower.layers.0.feed_forward1.ffw_layer_1.input_min, model.audio_tower.layers.0.feed_forward1.ffw_layer_1.linear.weight, model.audio_tower.layers.0.feed_forward1.ffw_layer_1.output_max, model.audio_tower.layers.0.feed_forward1.ffw_layer_1.output_min, model.audio_tower.layers.0.feed_forward1.ffw_layer_2.input_max, model.audio_tower.layers.0.feed_forward1.ffw_layer_2.input_min, model.audio_tower.layers.0.feed_forward1.ffw_layer_2.linear.weight, model.audio_tower.layers.0.feed_forward1.ffw_layer_2.output_max, model.audio_tower.layers.0.feed_forward1.ffw_layer_2.output_min, model.audio_tower.layers.0.feed_forward1.post_layer_norm.weight, model.audio_tower.layers.0.feed_forward1.pre_layer_norm.w

In [ ]:
import optax
from tunix.rl import rl_cluster as rl_cluster_lib
from tunix.rl.rollout import base_rollout

import os
os.environ["NEW_MODEL_DESIGN"] = "true"
optimizer = optax.adamw(learning_rate=1e-6,)

base_rollout_dict = {
    "max_prompt_length": 32,
    "kv_cache_size": 32 + 16 + 256,
    "temperature": 1.0,
    "top_p": 1.0,
    "top_k": 0,
    "return_logprobs": True,
    "max_tokens_to_generate": 16,
}

vllm_rollout_dict = {
    # vllm-tpu specific configs
    "rollout_vllm_model_version": model_version,
    "rollout_vllm_hbm_utilization": 0.33,
    "rollout_vllm_tpu_backend_type": "jax",
    "rollout_vllm_server_mode": True,
    "rollout_vllm_async_scheduling": True,
    "rollout_vllm_init_with_random_weights": False,
    "rollout_vllm_max_num_seqs": 1,
    "rollout_vllm_max_num_batched_tokens": 2496,
    "rollout_vllm_kwargs": {
        "kv_cache_metrics": True,
        "disable_log_stats": False,
        "enable_prefix_caching": False,
        "dtype": "bfloat16",
    },
}
rollout_engine_config = base_rollout.RolloutConfig(
    **base_rollout_dict, **vllm_rollout_dict
)
cluster_config = rl_cluster_lib.ClusterConfig(
    role_to_mesh={
        rl_cluster_lib.Role.ACTOR: trainer_mesh,
        rl_cluster_lib.Role.REFERENCE: trainer_mesh,
        rl_cluster_lib.Role.ROLLOUT: rollout_mesh,
    },
    rollout_engine="vllm",
    offload_to_cpu=False,
    training_config=rl_cluster_lib.RLTrainingConfig(
        actor_optimizer=optimizer,
        eval_every_n_steps=5,
    ),
    rollout_config=rollout_engine_config,
)

tokenizer = AutoTokenizer.from_pretrained(model_path)

rl_cluster = rl_cluster_lib.RLCluster(
    actor=trainer_model,
    reference=ref_model,
    tokenizer=tokenizer,
    cluster_config=cluster_config,
)

INFO 05-28 03:02:43 [__init__.py:59] TPU info: node_name=maxtext-single-host-1-v5p-8 | tpu_type=v5p-8 | worker_id=0 | num_chips=4 | num_cores_per_chip=2
2026-05-28 03:02:43 - WARNING - [absl] Tensorflow library not found, tensorflow.io.gfile operations will use native shim calls. GCS paths (i.e. 'gs://...') cannot be accessed.


/mnt/disks/linchai-data/miniconda3/envs/tunix/lib/python3.12/multiprocessing/popen_fork.py:66: RuntimeWarning: os.fork() was called. os.fork() is incompatible with multithreaded code, and JAX is multithreaded, so this will likely lead to a deadlock.
  self.pid = os.fork()
/mnt/disks/linchai-data/miniconda3/envs/tunix/lib/python3.12/multiprocessing/popen_fork.py:66: RuntimeWarning: os.fork() was called. os.fork() is incompatible with multithreaded code, and JAX is multithreaded, so this will likely lead to a deadlock.
  self.pid = os.fork()


INFO 05-28 03:02:46 [importing.py:46] Triton is installed but 0 active driver(s) found (expected 1). Disabling Triton to prevent runtime errors.
INFO 05-28 03:02:46 [importing.py:81] Triton not installed or not compatible; certain GPU-related functions will not be available.


2026-05-28 03:02:47,707	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


2026-05-28 03:02:48 - INFO - [absl] Engine kwargs setting key 'model' with value 'google/gemma-4-E2B-it'.
2026-05-28 03:02:48 - INFO - [absl] Engine kwargs setting key 'max_model_len' with value '304'.
2026-05-28 03:02:48 - INFO - [absl] Engine kwargs setting key 'async_scheduling' with value 'True'.
2026-05-28 03:02:48 - INFO - [absl] Engine kwargs setting key 'max_num_batched_tokens' with value '2496'.
2026-05-28 03:02:48 - INFO - [absl] Engine kwargs setting key 'max_num_seqs' with value '1'.
2026-05-28 03:02:48 - INFO - [absl] Engine kwargs setting key 'hf_config_path' with value 'None'.
2026-05-28 03:02:48 - INFO - [absl] Engine kwargs setting key 'max_logprobs' with value '1'.
2026-05-28 03:02:48 - INFO - [absl] Engine kwargs setting key 'logprobs_mode' with value 'processed_logprobs'.
2026-05-28 03:02:48 - INFO - [absl] Engine kwargs setting key 'kv_cache_metrics' with value 'True'.
2026-05-28 03:02:48 - INFO - [absl] Engine kwargs setting key 'disable_log_stats' with value 'Fal

2026-05-28 03:02:50 - WARNING - [huggingface_hub.utils._http] Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
WARNING 05-28 03:02:50 [arg_utils.py:1552] The global random seed is set to 0. Since VLLM_ENABLE_V1_MULTIPROCESSING is set to False, this may affect the random state of the Python process that launched vLLM.
INFO 05-28 03:02:50 [model.py:617] Resolved architecture: Gemma4ForConditionalGeneration
INFO 05-28 03:02:50 [model.py:1752] Using max model len 304
INFO 05-28 03:02:50 [scheduler.py:239] Chunked prefill is enabled with max_num_batched_tokens=2496.
WARNING 05-28 03:02:50 [scheduler.py:281] max_num_batched_tokens (2496) exceeds max_num_seqs * max_model_len (304). This may lead to unexpected behavior.
INFO 05-28 03:02:50 [config.py:100] Gemma4 model has heterogeneous head dimensions (head_dim=256, global_head_dim=512). Forcing TRITON_ATTN backend to prevent mixed-backend numerical diverg

ValidationError: 1 validation error for VllmConfig
  Assertion failed,  [type=assertion_error, input_value=ArgsKwargs((), {'model_co... 'shutdown_timeout': 0}), input_type=ArgsKwargs]
    For further information visit https://errors.pydantic.dev/2.13/v/assertion_error

In [ ]:
prompts = [{"role": "user", "content": "which is bigger, 1 or 2"}]
prompts = tokenizer.apply_chat_template(prompts, add_generation_prompt=True, tokenize=False)
out_data = rl_cluster.generate(prompts=prompts, max_generation_steps=16)
print("Text from VLLM sampler: ", out_data.text)
print("logprobs from VLLM sampler: ", out_data.logprobs)
print("tokens from VLLM sampler: ", out_data.tokens)
print("left_padded_prompt_tokens from VLLM sampler: ", out_data.left_padded_prompt_tokens)

In [ ]:
out_tokens = out_data.tokens[0]


prompt_tokens = out_data.left_padded_prompt_tokens[0]
completion_tokens = out_tokens
print(f"{prompt_tokens=}")
print(f"{completion_tokens=}")
prompt_tokens = jnp.repeat(jnp.array([prompt_tokens]), 4, axis=0)
completion_tokens = jnp.repeat(jnp.array([completion_tokens]), 4, axis=0)


from tunix.rl import common
graphdef, state = nnx.split(trainer_model)

batch_size = prompt_tokens.shape[0]
if batch_size == 0:
  raise ValueError(
      "Cannot get reference log probabilities from an empty batch."
  )

from tunix.sft import sharding_utils
with trainer_mesh:
    # This assumes reference model shards same data sharding as actor, which
    # should be true as ref model and policy model shares same architecture.
    dest_prompt_tokens = sharding_utils.shard_input(
        prompt_tokens,
         ("fsdp",),
    )
    dest_completion_tokens = sharding_utils.shard_input(
        completion_tokens,
         ("fsdp",),
    )
    dest_completion_mask = None

ref_logps = common.compute_per_token_logps(
    graphdef,
    state,
    prompt_tokens=dest_prompt_tokens,
    completion_tokens=dest_completion_tokens,
    pad_id=tokenizer.pad_token_id,
    eos_id=tokenizer.eos_token_id,
    temperature = 0.7,
)

In [ ]:
out_tokens = out_data.tokens[0]


prompt_tokens = out_data.padded_prompt_tokens[0]
completion_tokens = out_tokens
print(f"{prompt_tokens=}")
print(f"{completion_tokens=}")
prompt_tokens = jnp.repeat(jnp.array([prompt_tokens]), 4, axis=0)
completion_tokens = jnp.repeat(jnp.array([completion_tokens]), 4, axis=0)


from tunix.rl import common
graphdef, state = nnx.split(trainer_model)

batch_size = prompt_tokens.shape[0]
if batch_size == 0:
  raise ValueError(
      "Cannot get reference log probabilities from an empty batch."
  )

from tunix.sft import sharding_utils
with trainer_mesh:
    # This assumes reference model shards same data sharding as actor, which
    # should be true as ref model and policy model shares same architecture.
    dest_prompt_tokens = sharding_utils.shard_input(
        prompt_tokens,
         ("fsdp",),
    )
    dest_completion_tokens = sharding_utils.shard_input(
        completion_tokens,
         ("fsdp",),
    )
    dest_completion_mask = None

ref_logps = common.compute_per_token_logps(
    graphdef,
    state,
    prompt_tokens=dest_prompt_tokens,
    completion_tokens=dest_completion_tokens,
    pad_id=tokenizer.pad_token_id,
    eos_id=tokenizer.eos_token_id,
    temperature = 0.7,
)

In [ ]:
import jax
jax.clear_caches()
jax.block_until_ready(ref_logps)
print(f"{ref_logps.sharding = }")
print(f"{ref_logps[0] = }")

In [ ]:
logprobs_array = jnp.array(out_data.logprobs)
diff_mean = jnp.mean(jnp.abs(ref_logps - logprobs_array))
diff_std = jnp.std(jnp.abs(ref_logps - logprobs_array))
print(f"Mean absolute difference between reference log probabilities and VLLM sampler log probabilities: {diff_mean}")
print(f"Standard deviation of absolute difference: {diff_std}")